# Database Exploration

## Objective

The purpose of this notebook is to understand the structure of the MIMIC-III database before beginning data cleaning or machine learning.

Questions:

- What tables are available?
- What does each table represent?
- Which tables are relevant to mortality prediction?

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 

pd.set_option("display.max_columns", None)


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "mimic-iii-clinical-database-demo-1.4"
    / "mimic-iii-clinical-database-demo-1.4"
)

print(DATA_DIR)
print(DATA_DIR.exists())

/Users/tmali/Documents/Documents/MSc Healthcare Data Prep/Project - Predictive Health Analytics & Machine Learning with Python/data/raw/mimic-iii-clinical-database-demo-1.4/mimic-iii-clinical-database-demo-1.4
True


In [3]:
patients = pd.read_csv(DATA_DIR / "PATIENTS.csv")
patients.head()

,row_id,subject_id,gender,dob,dod,dod_hosp,dod_ssn,expire_flag
0,9467,10006,F,2094-03-05 00:00:00,2165-08-12 00:00:00,2165-08-12 00:00:00,2165-08-12 00:00:00,1
1,9472,10011,F,2090-06-05 00:00:00,2126-08-28 00:00:00,2126-08-28 00:00:00,NaN,1
2,9474,10013,F,2038-09-03 00:00:00,2125-10-07 00:00:00,2125-10-07 00:00:00,2125-10-07 00:00:00,1
3,9478,10017,F,2075-09-21 00:00:00,2152-09-12 00:00:00,NaN,2152-09-12 00:00:00,1
4,9479,10019,M,2114-06-20 00:00:00,2163-05-15 00:00:00,2163-05-15 00:00:00,2163-05-15 00:00:00,1


In [4]:
patients.shape

(100, 8)

In [5]:
patients.columns

Index(['row_id', 'subject_id', 'gender', 'dob', 'dod', 'dod_hosp', 'dod_ssn',
       'expire_flag'],
      dtype='str')

In [6]:
patients.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   row_id       100 non-null    int64
 1   subject_id   100 non-null    int64
 2   gender       100 non-null    str  
 3   dob          100 non-null    str  
 4   dod          100 non-null    str  
 5   dod_hosp     70 non-null     str  
 6   dod_ssn      77 non-null     str  
 7   expire_flag  100 non-null    int64
dtypes: int64(3), str(5)
memory usage: 6.4 KB


In [7]:
patients.describe()

,row_id,subject_id,expire_flag
count,100.000000,100.000000,100.0
mean,20452.580000,26162.330000,1.0
std,10982.043163,16201.834705,0.0
min,9467.000000,10006.000000,1.0
25%,9526.500000,10068.500000,1.0
50%,20209.000000,25128.000000,1.0
75%,31391.250000,42276.500000,1.0
max,31872.000000,44228.000000,1.0


### Initial Observations

- The PATIENTS table contains one row per patient.
- The primary key appears to be `subject_id`.
- The table records the date of birth and date of death of each patient.
- Not sure what 'dod_hosp' means, but there appear to be 30 missing entries.
- 'dod_ssn' has 23 missing entries. 
- The information contained in 'dod_hosp' and 'dod_ssn' appear to be identical to 'dod'.
- The purpose of 'expire_flag' is also unclear. The '1' value would have read to me as boolean, but 'patients.info()' show it as integer.  
- The table contains no obvious missing values in the ID fields.

In [8]:
patients["expire_flag"].value_counts()

expire_flag
1    100
Name: count, dtype: int64

In [9]:
patients["gender"].value_counts()

gender
F    55
M    45
Name: count, dtype: int64

In [10]:
patients["dod_hosp"].isna().sum()

np.int64(30)

### Observations Upon Investigation 

- The 'expire_flag' variable contains 100 instances of the value '1', meaning all entries are recorded as '1'. 
- expire_flag is a binary variable, represented numerically as 0/1 rather than using the Boolean True/False data type.
- The PATIENTS table contains a sex distribution of 55 entries for Female, and 45 entries for Male.

- Upon counting the sum of missing values for 'dod_hosp', it appears that there are 30 missing values for 'dod_hosp'. 
within the hospital database. 
- 'dod_ssn' refers to the date of death obtained from the Social Security database.
- The 30 missing entries for 'dod_hosp' would, then, indicate that those missing entries belonged to patients who's deaths were not recorded in the hospital database. These should not automatically be interpreted as deaths occurring outside the hospital, because death may instead have been identified through the Social Security death records ('dod_ssn'). The combined 'dod' variable incorporates information from both sources.

- 70% of deaths were recorded within the hospital database, and 30% were not.

In [11]:
admissions = pd.read_csv(DATA_DIR / "ADMISSIONS.csv")

In [12]:
admissions.head()

,row_id,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admission_location,discharge_location,insurance,language,religion,marital_status,ethnicity,edregtime,edouttime,diagnosis,hospital_expire_flag,has_chartevents_data
0,12258,10006,142345,2164-10-23 21:09:00,2164-11-01 17:15:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,SEPARATED,BLACK/AFRICAN AMERICAN,2164-10-23 16:43:00,2164-10-23 23:00:00,SEPSIS,0,1
1,12263,10011,105331,2126-08-14 22:32:00,2126-08-28 18:59:00,2126-08-28 18:59:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Private,NaN,CATHOLIC,SINGLE,UNKNOWN/NOT SPECIFIED,NaN,NaN,HEPATITIS B,1,1
2,12265,10013,165520,2125-10-04 23:36:00,2125-10-07 15:13:00,2125-10-07 15:13:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,NaN,UNKNOWN/NOT SPECIFIED,NaN,NaN,SEPSIS,1,1
3,12269,10017,199207,2149-05-26 17:19:00,2149-06-03 18:42:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,SNF,Medicare,NaN,CATHOLIC,DIVORCED,WHITE,2149-05-26 12:08:00,2149-05-26 19:45:00,HUMERAL FRACTURE,0,1
4,12270,10019,177759,2163-05-14 20:43:00,2163-05-15 12:00:00,2163-05-15 12:00:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,DIVORCED,WHITE,NaN,NaN,ALCOHOLIC HEPATITIS,1,1


In [13]:
admissions.shape

(129, 19)

In [14]:
admissions.columns

Index(['row_id', 'subject_id', 'hadm_id', 'admittime', 'dischtime',
       'deathtime', 'admission_type', 'admission_location',
       'discharge_location', 'insurance', 'language', 'religion',
       'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis',
       'hospital_expire_flag', 'has_chartevents_data'],
      dtype='str')

In [15]:
admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   row_id                129 non-null    int64
 1   subject_id            129 non-null    int64
 2   hadm_id               129 non-null    int64
 3   admittime             129 non-null    str  
 4   dischtime             129 non-null    str  
 5   deathtime             40 non-null     str  
 6   admission_type        129 non-null    str  
 7   admission_location    129 non-null    str  
 8   discharge_location    129 non-null    str  
 9   insurance             129 non-null    str  
 10  language              81 non-null     str  
 11  religion              128 non-null    str  
 12  marital_status        113 non-null    str  
 13  ethnicity             129 non-null    str  
 14  edregtime             92 non-null     str  
 15  edouttime             92 non-null     str  
 16  diagnosis          

## Questions Raised

- What is the distribution of `hospital_expire_flag`?
- What are the different `admission_type` categories?
- What are the different `admission_location` categories?
- Why are `edregtime` and `edouttime` missing for some admissions?
- What categories are present in `religion`?
- Which variables are appropriate predictors of mortality?

In [16]:
admissions["hospital_expire_flag"].value_counts()

hospital_expire_flag
0    89
1    40
Name: count, dtype: int64

In [17]:
admissions["hospital_expire_flag"].value_counts(normalize=True)

hospital_expire_flag
0    0.689922
1    0.310078
Name: proportion, dtype: float64

In [18]:
admissions["admission_type"].value_counts()

admission_type
EMERGENCY    119
ELECTIVE       8
URGENT         2
Name: count, dtype: int64

In [19]:
admissions["admission_type"].value_counts(normalize=True)

admission_type
EMERGENCY    0.922481
ELECTIVE     0.062016
URGENT       0.015504
Name: proportion, dtype: float64

In [20]:
admissions["religion"].value_counts()

religion
CATHOLIC               59
NOT SPECIFIED          19
UNOBTAINABLE           19
PROTESTANT QUAKER      10
JEWISH                 10
OTHER                   4
CHRISTIAN SCIENTIST     3
MUSLIM                  2
BUDDHIST                1
ROMANIAN EAST. ORTH     1
Name: count, dtype: int64

In [21]:
admissions["religion"].value_counts(normalize=True)

religion
CATHOLIC               0.460938
NOT SPECIFIED          0.148438
UNOBTAINABLE           0.148438
PROTESTANT QUAKER      0.078125
JEWISH                 0.078125
OTHER                  0.031250
CHRISTIAN SCIENTIST    0.023438
MUSLIM                 0.015625
BUDDHIST               0.007812
ROMANIAN EAST. ORTH    0.007812
Name: proportion, dtype: float64

## Target Variable Investigation

The `hospital_expire_flag` variable represents whether a patient died during a particular hospital admission.

- 89 admissions (68.99%) had a value of 0, indicating that the patient did not die during the admission.
- 40 admissions (31.01%) had a value of 1, indicating that the patient died during the admission.
- The target variable is therefore moderately imbalanced, but the minority class still represents approximately 31% of observations.
- A majority-class baseline would achieve approximately 69% accuracy by predicting survival for every admission, while failing to identify any deaths.
- Therefore, accuracy alone would not provide an adequate assessment of model performance.

The `ADMISSION_TYPE` variable is highly dominated by emergency admissions, which account for approximately 92.2% of observations in the Demo dataset.

In [22]:
icustays = pd.read_csv(DATA_DIR / "ICUSTAYS.csv")

In [23]:
icustays.head()

,row_id,subject_id,hadm_id,icustay_id,dbsource,first_careunit,last_careunit,first_wardid,last_wardid,intime,outtime,los
0,12742,10006,142345,206504,carevue,MICU,MICU,52,52,2164-10-23 21:10:15,2164-10-25 12:21:07,1.6325
1,12747,10011,105331,232110,carevue,MICU,MICU,15,15,2126-08-14 22:34:00,2126-08-28 18:59:00,13.8507
2,12749,10013,165520,264446,carevue,MICU,MICU,15,15,2125-10-04 23:38:00,2125-10-07 15:13:52,2.6499
3,12754,10017,199207,204881,carevue,CCU,CCU,7,7,2149-05-29 18:52:29,2149-05-31 22:19:17,2.1436
4,12755,10019,177759,228977,carevue,MICU,MICU,15,15,2163-05-14 20:43:56,2163-05-16 03:47:04,1.2938


In [24]:
icustays.shape 

(136, 12)

In [25]:
icustays.columns

Index(['row_id', 'subject_id', 'hadm_id', 'icustay_id', 'dbsource',
       'first_careunit', 'last_careunit', 'first_wardid', 'last_wardid',
       'intime', 'outtime', 'los'],
      dtype='str')

In [26]:
icustays.info()

<class 'pandas.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   row_id          136 non-null    int64  
 1   subject_id      136 non-null    int64  
 2   hadm_id         136 non-null    int64  
 3   icustay_id      136 non-null    int64  
 4   dbsource        136 non-null    str    
 5   first_careunit  136 non-null    str    
 6   last_careunit   136 non-null    str    
 7   first_wardid    136 non-null    int64  
 8   last_wardid     136 non-null    int64  
 9   intime          136 non-null    str    
 10  outtime         136 non-null    str    
 11  los             136 non-null    float64
dtypes: float64(1), int64(6), str(5)
memory usage: 12.9 KB


In [27]:
icustays["subject_id"].value_counts()

subject_id
41976    15
10124     4
10088     3
44083     3
10059     2
         ..
43927     1
44154     1
44212     1
44222     1
44228     1
Name: count, Length: 100, dtype: int64

In [28]:
icustays["subject_id"].nunique()

100

In [29]:
admissions["subject_id"].nunique()

100

In [30]:
diagnoses = pd.read_csv(DATA_DIR / "DIAGNOSES_ICD.csv")

In [31]:
diagnoses.head()

,row_id,subject_id,hadm_id,seq_num,icd9_code
0,112344,10006,142345,1,99591
1,112345,10006,142345,2,99662
2,112346,10006,142345,3,5672
3,112347,10006,142345,4,40391
4,112348,10006,142345,5,42731


In [32]:
diagnoses.shape

(1761, 5)

In [33]:
diagnoses.columns

Index(['row_id', 'subject_id', 'hadm_id', 'seq_num', 'icd9_code'], dtype='str')

In [34]:
diagnoses.info()

<class 'pandas.DataFrame'>
RangeIndex: 1761 entries, 0 to 1760
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   row_id      1761 non-null   int64
 1   subject_id  1761 non-null   int64
 2   hadm_id     1761 non-null   int64
 3   seq_num     1761 non-null   int64
 4   icd9_code   1761 non-null   str  
dtypes: int64(4), str(1)
memory usage: 68.9 KB


In [35]:
diagnoses["hadm_id"].nunique()

129

In [36]:
diagnoses["subject_id"].nunique()

100

In [37]:
icd_dictionary = pd.read_csv(DATA_DIR / "D_ICD_DIAGNOSES.csv")

In [38]:
icd_dictionary.head()

,row_id,icd9_code,short_title,long_title
0,1,01716,Erythem nod tb-oth test,Erythema nodosum with hypersensitivity reactio...
1,2,01720,TB periph lymph-unspec,"Tuberculosis of peripheral lymph nodes, unspec..."
2,3,01721,TB periph lymph-no exam,"Tuberculosis of peripheral lymph nodes, bacter..."
3,4,01722,TB periph lymph-exam unk,"Tuberculosis of peripheral lymph nodes, bacter..."
4,5,01723,TB periph lymph-micro dx,"Tuberculosis of peripheral lymph nodes, tuberc..."


In [39]:
icd_dictionary.shape

(14567, 4)

In [40]:
icd_dictionary.columns

Index(['row_id', 'icd9_code', 'short_title', 'long_title'], dtype='str')

In [41]:
icd_dictionary.info()

<class 'pandas.DataFrame'>
RangeIndex: 14567 entries, 0 to 14566
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   row_id       14567 non-null  int64
 1   icd9_code    14567 non-null  str  
 2   short_title  14567 non-null  str  
 3   long_title   14567 non-null  str  
dtypes: int64(1), str(3)
memory usage: 455.3 KB


In [42]:
diagnoses["icd9_code"].iloc[0]

'99591'

In [43]:
icd_dictionary[icd_dictionary["icd9_code"] == "99591"]

,row_id,icd9_code,short_title,long_title
11402,11403,99591,Sepsis,Sepsis


In [44]:
labevents = pd.read_csv(DATA_DIR / "LABEVENTS.csv")

In [45]:
labevents.head()

,row_id,subject_id,hadm_id,itemid,charttime,value,valuenum,valueuom,flag
0,6244563,10006,NaN,50868,2164-09-24 20:21:00,19,19.0,mEq/L,NaN
1,6244564,10006,NaN,50882,2164-09-24 20:21:00,27,27.0,mEq/L,NaN
2,6244565,10006,NaN,50893,2164-09-24 20:21:00,10.0,10.0,mg/dL,NaN
3,6244566,10006,NaN,50902,2164-09-24 20:21:00,97,97.0,mEq/L,NaN
4,6244567,10006,NaN,50912,2164-09-24 20:21:00,7.0,7.0,mg/dL,abnormal


In [46]:
labevents.shape

(76074, 9)

In [47]:
labevents.columns

Index(['row_id', 'subject_id', 'hadm_id', 'itemid', 'charttime', 'value',
       'valuenum', 'valueuom', 'flag'],
      dtype='str')

In [48]:
labevents.info()

<class 'pandas.DataFrame'>
RangeIndex: 76074 entries, 0 to 76073
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   row_id      76074 non-null  int64  
 1   subject_id  76074 non-null  int64  
 2   hadm_id     61812 non-null  float64
 3   itemid      76074 non-null  int64  
 4   charttime   76074 non-null  str    
 5   value       76069 non-null  str    
 6   valuenum    67030 non-null  float64
 7   valueuom    66669 non-null  str    
 8   flag        29737 non-null  str    
dtypes: float64(2), int64(3), str(4)
memory usage: 5.2 MB


In [49]:
labevents["itemid"].nunique()

439

In [50]:
labevents["hadm_id"].nunique()

129

In [51]:
labevents["subject_id"].nunique()

100

In [52]:
dlabitems = pd.read_csv(DATA_DIR / "D_LABITEMS.csv")

In [53]:
dlabitems.head()

,row_id,itemid,label,fluid,category,loinc_code
0,1,50800,SPECIMEN TYPE,BLOOD,BLOOD GAS,NaN
1,2,50801,Alveolar-arterial Gradient,Blood,Blood Gas,19991-9
2,3,50802,Base Excess,Blood,Blood Gas,11555-0
3,4,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas,1959-6
4,5,50804,Calculated Total CO2,Blood,Blood Gas,34728-6


In [54]:
dlabitems.shape

(753, 6)

In [55]:
dlabitems.columns

Index(['row_id', 'itemid', 'label', 'fluid', 'category', 'loinc_code'], dtype='str')

In [56]:
dlabitems.info()

<class 'pandas.DataFrame'>
RangeIndex: 753 entries, 0 to 752
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   row_id      753 non-null    int64
 1   itemid      753 non-null    int64
 2   label       753 non-null    str  
 3   fluid       753 non-null    str  
 4   category    753 non-null    str  
 5   loinc_code  585 non-null    str  
dtypes: int64(2), str(4)
memory usage: 35.4 KB


In [57]:
labevents["hadm_id"].isna().sum()

np.int64(14262)

In [58]:
labevents["hadm_id"].isna().mean()

np.float64(0.1874753529458159)

In [59]:
labevents["subject_id"].isna().sum()

np.int64(0)

In [60]:
labevents[labevents["hadm_id"].isna()]["subject_id"].nunique()

47

In [61]:
labevents["valuenum"].describe()

count    67030.000000
mean        76.745849
std        634.732230
min        -29.000000
25%          4.200000
50%         18.000000
75%         60.000000
max      52950.000000
Name: valuenum, dtype: float64

In [62]:
labevents["valuenum"].isna().sum()

np.int64(9044)

In [63]:
labevents["valuenum"].isna().mean()

np.float64(0.11888424428845598)

In [64]:
top_items = labevents["itemid"].value_counts().head(10).index

In [65]:
dlabitems[dlabitems["itemid"].isin(top_items)]

,row_id,itemid,label,fluid,category,loinc_code
68,69,50868,Anion Gap,Blood,Chemistry,1863-0
82,83,50882,Bicarbonate,Blood,Chemistry,1963-8
102,103,50902,Chloride,Blood,Chemistry,2075-0
112,113,50912,Creatinine,Blood,Chemistry,2160-0
131,132,50931,Glucose,Blood,Chemistry,2345-7
171,172,50971,Potassium,Blood,Chemistry,2823-3
183,184,50983,Sodium,Blood,Chemistry,2951-2
205,206,51006,Urea Nitrogen,Blood,Chemistry,3094-0
420,421,51221,Hematocrit,Blood,Hematology,4544-3
464,465,51265,Platelet Count,Blood,Hematology,777-3


In [66]:
labevents["itemid"].value_counts(normalize=True).head(10)

itemid
51221    0.030457
50971    0.029958
50983    0.028722
50912    0.028591
50902    0.028393
51006    0.028367
50882    0.028275
50868    0.028052
50931    0.027881
51265    0.027447
Name: proportion, dtype: float64

In [67]:
labevents["itemid"].value_counts(normalize=True).head(10).sum()

np.float64(0.28614244025554064)

## Observation 
- The ten most frequently recorded laboratory tests accounted for approximately 28.6% of all recorded lab events, indicating that laboratory activity was distributed across a relatively diverse range of test types.

In [68]:
patients["gender"].value_counts()

gender
F    55
M    45
Name: count, dtype: int64

In [69]:
patients["gender"].value_counts(normalize=True)

gender
F    0.55
M    0.45
Name: proportion, dtype: float64

In [70]:
patients["dob"].min()

'1844-07-18 00:00:00'

In [71]:
patients["dob"].max()

'2181-04-19 00:00:00'

In [72]:
admissions["admission_type"].value_counts()

admission_type
EMERGENCY    119
ELECTIVE       8
URGENT         2
Name: count, dtype: int64

In [73]:
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"])
admissions["admittime"] = pd.to_datetime(admissions["admittime"])

In [74]:
admissions["length_of_stay"] = (
    admissions["dischtime"] - admissions["admittime"]
).dt.total_seconds() / (24 * 60 * 60)

In [75]:
admissions["length_of_stay"].describe()

count    129.000000
mean       9.332332
std       12.720450
min        0.038194
25%        3.314583
50%        6.634028
75%       10.645139
max      123.984722
Name: length_of_stay, dtype: float64

In [76]:
admissions.groupby("hospital_expire_flag")["length_of_stay"].describe()

,count,mean,std,min,25%,50%,75%,max
hospital_expire_flag,,,,,,,,
0,89.0,9.766526,13.934030,0.980556,4.104167,6.830556,10.146528,123.984722
1,40.0,8.366250,9.569871,0.038194,1.882986,4.819444,12.564063,35.248611


In [77]:
admissions.groupby("hospital_expire_flag")["admission_type"].value_counts()

hospital_expire_flag  admission_type
0                     EMERGENCY         80
                      ELECTIVE           8
                      URGENT             1
1                     EMERGENCY         39
                      URGENT             1
Name: count, dtype: int64

 ## Observations  
 
 Most admission types regardless of whether the patient died or not were an emergency admission. 
 
 Elective admission types were true for 8 admissions where the patient survived, and none for deaths.

In [78]:
admissions_with_gender = admissions[["subject_id", "hadm_id", "hospital_expire_flag"]].merge(
    patients[["subject_id", "gender"]],
    on = "subject_id"
)


In [79]:
admissions_with_gender.groupby("hospital_expire_flag")["gender"].value_counts()

hospital_expire_flag  gender
0                     M         51
                      F         38
1                     F         21
                      M         19
Name: count, dtype: int64

In [80]:
diagnoses_per_admission = diagnoses.groupby("hadm_id").size()

In [81]:
diagnoses_per_admission.describe()

count    129.000000
mean      13.651163
std        6.465837
min        3.000000
25%        9.000000
50%       12.000000
75%       17.000000
max       37.000000
dtype: float64

In [82]:
diagnoses_per_admission.sort_values(ascending=False).head(10)

hadm_id
153826    37
186361    36
101361    31
126949    27
163189    24
149469    24
195911    24
116518    24
167754    23
112662    23
dtype: int64

In [83]:
diagnoses["icd9_code"].value_counts().head(10)

icd9_code
4019     53
42731    48
5849     45
4280     39
25000    31
51881    31
2724     29
5990     27
486      26
99592    25
Name: count, dtype: int64

In [84]:
top_diagnoses = diagnoses["icd9_code"].value_counts().head(10).index

icd_dictionary[icd_dictionary["icd9_code"].isin(top_diagnoses)]

,row_id,icd9_code,short_title,long_title
1590,1591,25000,DMII wo cmp nt st uncntr,Diabetes mellitus without mention of complicat...
2746,2747,2724,Hyperlipidemia NEC/NOS,Other and unspecified hyperlipidemia
4303,4304,4019,Hypertension NOS,Unspecified essential hypertension
4461,4462,42731,Atrial fibrillation,Atrial fibrillation
4472,4473,4280,CHF NOS,"Congestive heart failure, unspecified"
5278,5279,51881,Acute respiratry failure,Acute respiratory failure
5527,5528,486,"Pneumonia, organism NOS","Pneumonia, organism unspecified"
5907,5908,5849,Acute kidney failure NOS,"Acute kidney failure, unspecified"
6375,6376,5990,Urin tract infection NOS,"Urinary tract infection, site not specified"
11403,11404,99592,Severe sepsis,Severe sepsis


In [85]:
diagnoses[diagnoses["icd9_code"] == "4019"]["hadm_id"].nunique()

53

In [86]:
diagnoses[diagnoses["icd9_code"] == "4019"]["hadm_id"].nunique() / admissions["hadm_id"].nunique()

0.4108527131782946

In [87]:
hypertension_admissions = diagnoses[
    diagnoses["icd9_code"] == "4019"
]["hadm_id"].unique()

In [88]:
patients["dob"].head()

0    2094-03-05 00:00:00
1    2090-06-05 00:00:00
2    2038-09-03 00:00:00
3    2075-09-21 00:00:00
4    2114-06-20 00:00:00
Name: dob, dtype: str

In [89]:
patients["dob"].describe()

count                     100
unique                     99
top       2058-04-23 00:00:00
freq                        2
Name: dob, dtype: object

In [90]:
patients[["subject_id", "dob", "expire_flag"]].head(10)

,subject_id,dob,expire_flag
0,10006,2094-03-05 00:00:00,1
1,10011,2090-06-05 00:00:00,1
2,10013,2038-09-03 00:00:00,1
3,10017,2075-09-21 00:00:00,1
4,10019,2114-06-20 00:00:00,1
5,10026,1895-05-17 00:00:00,1
6,10027,2108-01-15 00:00:00,1
7,10029,2061-04-10 00:00:00,1
8,10032,2050-03-29 00:00:00,1
9,10033,2051-04-21 00:00:00,1


In [91]:
patients["expire_flag"].value_counts()

expire_flag
1    100
Name: count, dtype: int64

In [92]:
patients["dob"].str[:4].value_counts().sort_index()

dob
1844    1
1846    1
1851    1
1876    1
1878    1
       ..
2136    2
2141    1
2146    1
2150    1
2181    1
Name: count, Length: 66, dtype: int64